# 03 - Baseline: Lexicon, TF-IDF + LR, dan Kata Kunci (Fase 6)

## Setup

In [1]:
!pip install -q scikit-learn joblib

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import hashlib
import time
import urllib.request
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_fscore_support
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import FeatureUnion, Pipeline

FOLDER_PROJECT = Path('/content/drive/MyDrive/ulasan-ewallet')
FOLDER_PROCESSED = FOLDER_PROJECT / 'data' / 'processed'
FOLDER_GOLD = FOLDER_PROJECT / 'data' / 'gold'
FOLDER_KAMUS = FOLDER_PROJECT / 'kamus'
FOLDER_MODEL = FOLDER_PROJECT / 'models'
FOLDER_MODEL.mkdir(parents=True, exist_ok=True)
SEED = 42

ASPEK = ['transaksi', 'keamanan', 'akun', 'error_performa', 'cs', 'biaya_promo', 'paylater', 'kemudahan']
KELAS = ['neg', 'net', 'pos']
RATING_KE_SENTIMEN = {1: 'neg', 2: 'neg', 3: 'net', 4: 'pos', 5: 'pos'}

ulasan = pd.read_parquet(
    FOLDER_PROCESSED / 'ulasan_bersih.parquet',
    columns=['id_ulasan', 'aplikasi', 'rating', 'teks_ringan', 'teks_penuh', 'jumlah_kata'],
)
gold_train = pd.read_parquet(FOLDER_GOLD / 'gold_train.parquet').merge(
    ulasan[['id_ulasan', 'teks_ringan', 'teks_penuh', 'jumlah_kata']], on='id_ulasan', how='left'
)
assert gold_train['teks_penuh'].notna().all(), 'Ada ulasan gold yang tidak ditemukan di ulasan_bersih'
# Versi label (sama dengan 04_indobert): menandai ringkasan dev dibuat dari label gold train yang mana
VERSI_LABEL = hashlib.md5(
    gold_train.sort_values('id_label')[['id_label', 'sentimen'] + ASPEK].to_csv(index=False).encode()
).hexdigest()[:8]
dev = gold_train[gold_train['sumber'] != 'tambahan'].reset_index(drop=True)
tambahan = gold_train[gold_train['sumber'] == 'tambahan'].reset_index(drop=True)

# Data berlabel rating: seluruh ulasan bersih dikurangi SEMUA ulasan yang pernah dilabel (test, train, pilot, tambahan)
id_berlabel = set()
for nama_file in ['gold_kunci.parquet', 'pilot_kunci.parquet', 'tambahan_kunci.parquet']:
    id_berlabel |= set(pd.read_parquet(FOLDER_GOLD / nama_file)['id_ulasan'])
lemah = ulasan[~ulasan['id_ulasan'].isin(id_berlabel)].reset_index(drop=True)
lemah['sentimen'] = lemah['rating'].map(RATING_KE_SENTIMEN)

print(f'versi label: {VERSI_LABEL} | dev: {len(dev)} | tambahan: {len(tambahan)} | berlabel rating: {len(lemah):,}')
print('Sentimen dev:', dev['sentimen'].value_counts().to_dict())
print('Sentimen berlabel rating (%):', lemah['sentimen'].value_counts(normalize=True).mul(100).round(1).to_dict())

versi label: 6b6ec0d2 | dev: 855 | tambahan: 153 | berlabel rating: 397,434
Sentimen dev: {'neg': 586, 'pos': 210, 'net': 59}
Sentimen berlabel rating (%): {'pos': 63.7, 'neg': 31.6, 'net': 4.7}


In [4]:
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=KELAS, average='macro', zero_division=0)


def interval_bootstrap(y_true, y_pred, fungsi, n=1000):
    """Interval kepercayaan 95% dengan resampling baris (berlaku untuk label tunggal maupun multi-label)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    indeks = np.random.default_rng(SEED).integers(0, len(y_true), size=(n, len(y_true)))
    return np.percentile([fungsi(y_true[i], y_pred[i]) for i in indeks], [2.5, 97.5])


hasil_sentimen = []


def catat_sentimen(nama, y_pred, catatan=''):
    """Nilai prediksi sentimen di dev, simpan ke tabel ringkasan, tampilkan confusion matrix."""
    y_true, y_pred = dev['sentimen'].to_numpy(), np.asarray(y_pred)
    skor = macro_f1(y_true, y_pred)
    bawah, atas = interval_bootstrap(y_true, y_pred, macro_f1)
    f1_kelas = f1_score(y_true, y_pred, labels=KELAS, average=None, zero_division=0)
    hasil_sentimen[:] = [h for h in hasil_sentimen if h['model'] != nama]
    hasil_sentimen.append({
        'model': nama, 'macro_f1': round(skor, 3), 'ci_95': f'{bawah:.2f}-{atas:.2f}',
        **{f'f1_{k}': round(v, 3) for k, v in zip(KELAS, f1_kelas)}, 'catatan': catatan,
    })
    print(f'{nama}: macro-F1 {skor:.3f} (CI 95% {bawah:.2f}-{atas:.2f})')
    display(pd.crosstab(y_true, y_pred, rownames=['gold'], colnames=['prediksi']))


def buat_vektorisasi(min_df):
    """TF-IDF kata (1-2 gram) + karakter (3-5 gram). Karakter membantu untuk salah ketik dan slang."""
    return FeatureUnion([
        ('kata', TfidfVectorizer(ngram_range=(1, 2), min_df=min_df, max_features=200_000, sublinear_tf=True)),
        ('huruf', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=min_df,
                                  max_features=200_000, sublinear_tf=True)),
    ])


lipatan = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# A. Sentimen
## 6.1 Rating bintang sebagai label

In [5]:
catat_sentimen('Rating bintang', dev['rating'].map(RATING_KE_SENTIMEN), 'bukan model; label rating vs label manual')

Rating bintang: macro-F1 0.523 (CI 95% 0.49-0.55)


prediksi,neg,net,pos
gold,,,
neg,329,134,123
net,9,20,30
pos,4,17,189


## 6.2 Panjang ulasan saja

In [6]:
X_panjang = np.log1p(dev[['jumlah_kata']].to_numpy())
pred = np.empty(len(dev), dtype=object)
for latih, uji in lipatan.split(dev, dev['sentimen']):
    model = LogisticRegression(class_weight='balanced').fit(X_panjang[latih], dev['sentimen'].iloc[latih])
    pred[uji] = model.predict(X_panjang[uji])
catat_sentimen('Panjang ulasan saja', pred, 'LR 1 fitur, 5-fold di dev')

Panjang ulasan saja: macro-F1 0.534 (CI 95% 0.50-0.57)


prediksi,neg,net,pos
gold,,,
neg,397,132,57
net,16,18,25
pos,19,41,150


## 6.3 Lexicon InSet (tanpa training)

In [7]:
URL_INSET = 'https://raw.githubusercontent.com/fajri91/InSet/master/{}.tsv'
leksikon = {}
for jenis in ['positive', 'negative']:
    path = FOLDER_KAMUS / f'inset_{jenis}.tsv'
    if not path.exists():
        urllib.request.urlretrieve(URL_INSET.format(jenis), path)
    tabel = pd.read_csv(path, sep='\t')
    for kata, bobot in zip(tabel.iloc[:, 0].astype(str).str.lower().str.strip(), tabel.iloc[:, 1].astype(float)):
        leksikon[kata] = leksikon.get(kata, 0.0) + bobot
print(f'InSet: {len(leksikon):,} entri')

NEGASI = {'tidak', 'bukan', 'belum', 'kurang', 'tak', 'jangan', 'tanpa'}


def skor_inset(teks):
    token = teks.split()
    skor, i = 0.0, 0
    while i < len(token):
        bigram = ' '.join(token[i:i + 2])
        if i + 1 < len(token) and bigram in leksikon:
            nilai, langkah = leksikon[bigram], 2
        else:
            nilai, langkah = leksikon.get(token[i], 0.0), 1
        if i > 0 and token[i - 1] in NEGASI:
            nilai = -nilai
        skor += nilai
        i += langkah
    return skor


skor_dev = dev['teks_ringan'].map(skor_inset).to_numpy()
print('Persen ulasan dev dengan skor 0 (tidak ada kata InSet):', round((skor_dev == 0).mean() * 100, 1))


def label_dari_skor(skor, t_neg, t_pos):
    return np.where(skor <= t_neg, 'neg', np.where(skor > t_pos, 'pos', 'net'))


kandidat = np.unique(np.percentile(skor_dev, np.arange(1, 100)))
terbaik = max(
    ((macro_f1(dev['sentimen'], label_dari_skor(skor_dev, a, b)), a, b)
     for i, a in enumerate(kandidat) for b in kandidat[i:]),
    key=lambda x: x[0],
)
_, T_NEG, T_POS = terbaik
print(f'Ambang terpilih: neg jika skor <= {T_NEG:.1f}, pos jika skor > {T_POS:.1f}')
catat_sentimen('Lexicon InSet', label_dari_skor(skor_dev, T_NEG, T_POS), 'ambang dipilih di dev')

InSet: 9,071 entri
Persen ulasan dev dengan skor 0 (tidak ada kata InSet): 7.4
Ambang terpilih: neg jika skor <= -1.0, pos jika skor > 1.0
Lexicon InSet: macro-F1 0.467 (CI 95% 0.43-0.51)


prediksi,neg,net,pos
gold,,,
neg,470,46,70
net,22,14,23
pos,96,34,80


## 6.4 TF-IDF + LR, label rating

In [8]:
mulai = time.time()
vektor_lemah = buat_vektorisasi(min_df=5)
X_lemah = vektor_lemah.fit_transform(lemah['teks_penuh'])
X_dev = vektor_lemah.transform(dev['teks_penuh'])
print(f'Fitur: {X_lemah.shape[1]:,} | {(time.time() - mulai) / 60:.1f} menit')

model_lemah, skor_lemah = {}, {}
for C in [0.25, 1, 4]:
    model_lemah[C] = LogisticRegression(C=C, class_weight='balanced', max_iter=1000).fit(X_lemah, lemah['sentimen'])
    skor_lemah[C] = macro_f1(dev['sentimen'], model_lemah[C].predict(X_dev))
    print(f'C={C}: macro-F1 dev {skor_lemah[C]:.3f} | {(time.time() - mulai) / 60:.1f} menit')

C_LEMAH = max(skor_lemah, key=skor_lemah.get)
catat_sentimen(f'TF-IDF + LR, label rating (C={C_LEMAH})', model_lemah[C_LEMAH].predict(X_dev),
               f'{len(lemah):,} ulasan berlabel rating')
joblib.dump(Pipeline([('vektor', vektor_lemah), ('model', model_lemah[C_LEMAH])]),
            FOLDER_MODEL / 'tfidf_sentimen_rating.joblib')

Fitur: 127,154 | 0.8 menit
C=0.25: macro-F1 dev 0.598 | 3.6 menit
C=1: macro-F1 dev 0.599 | 7.0 menit
C=4: macro-F1 dev 0.595 | 13.0 menit
TF-IDF + LR, label rating (C=1): macro-F1 0.599 (CI 95% 0.57-0.63)


prediksi,neg,net,pos
gold,,,
neg,396,152,38
net,7,22,30
pos,4,13,193


['/content/drive/MyDrive/ulasan-ewallet/models/tfidf_sentimen_rating.joblib']

## 6.5 TF-IDF + LR, label gold

In [9]:
def model_sentimen_gold(C):
    return Pipeline([
        ('vektor', buat_vektorisasi(min_df=2)),
        ('model', LogisticRegression(C=C, class_weight='balanced', max_iter=2000)),
    ])


pred_gold, skor_gold = {}, {}
for C in [1, 4, 16]:
    pred = np.empty(len(dev), dtype=object)
    for latih, uji in lipatan.split(dev, dev['sentimen']):
        data_latih = pd.concat([dev.iloc[latih], tambahan])
        model = model_sentimen_gold(C).fit(data_latih['teks_penuh'], data_latih['sentimen'])
        pred[uji] = model.predict(dev['teks_penuh'].iloc[uji])
    pred_gold[C], skor_gold[C] = pred, macro_f1(dev['sentimen'], pred)
    print(f'C={C}: macro-F1 5-fold {skor_gold[C]:.3f}')

C_GOLD = max(skor_gold, key=skor_gold.get)
catat_sentimen(f'TF-IDF + LR, label gold (C={C_GOLD})', pred_gold[C_GOLD], '5-fold di dev + sampel terarah')

# Model final dilatih dengan seluruh gold train, untuk dinilai di test nanti
joblib.dump(model_sentimen_gold(C_GOLD).fit(gold_train['teks_penuh'], gold_train['sentimen']),
            FOLDER_MODEL / 'tfidf_sentimen_gold.joblib')

C=1: macro-F1 5-fold 0.817
C=4: macro-F1 5-fold 0.813
C=16: macro-F1 5-fold 0.812
TF-IDF + LR, label gold (C=1): macro-F1 0.817 (CI 95% 0.77-0.85)


prediksi,neg,net,pos
gold,,,
neg,550,20,16
net,22,35,2
pos,17,2,191


['/content/drive/MyDrive/ulasan-ewallet/models/tfidf_sentimen_gold.joblib']

## 6.6 Ringkasan sentimen (dev)

In [10]:
ringkasan_sentimen = pd.DataFrame(hasil_sentimen).assign(versi_label=VERSI_LABEL)
ringkasan_sentimen.to_parquet(FOLDER_PROCESSED / 'ringkasan_dev_baseline_sentimen.parquet', index=False)  # dibaca 04
ringkasan_sentimen.set_index('model')

,macro_f1,ci_95,f1_neg,f1_net,f1_pos,catatan,versi_label
model,,,,,,,
Rating bintang,0.523,0.49-0.55,0.709,0.174,0.685,bukan model; label rating vs label manual,6b6ec0d2
Panjang ulasan saja,0.534,0.50-0.57,0.780,0.144,0.679,"LR 1 fitur, 5-fold di dev",6b6ec0d2
Lexicon InSet,0.467,0.43-0.51,0.801,0.183,0.418,ambang dipilih di dev,6b6ec0d2
"TF-IDF + LR, label rating (C=1)",0.599,0.57-0.63,0.798,0.179,0.820,"397,434 ulasan berlabel rating",6b6ec0d2
"TF-IDF + LR, label gold (C=1)",0.817,0.77-0.85,0.936,0.603,0.912,5-fold di dev + sampel terarah,6b6ec0d2


# B. Aspek (multi-label)
## 6.7 Aturan kata kunci vs TF-IDF + One-vs-Rest LR

In [11]:
POLA_ASPEK = {
    'transaksi': r'topup|transfer|tarik|qris|bayar|pembayaran|pulsa|paket data|token|kirim uang|refund|pending'
                 r'|belum masuk|tidak masuk|saldo masuk',
    'keamanan': r'saldo (?:saya |ku |nya )?(?:hilang|berkurang)|hilang sendiri|berkurang sendiri|tidak di ?kenal'
                r'|tanpa sepengetahuan|bobol|retas|hack|penipu|langganan|google play|transaksi sendiri'
                r'|\baman\b|terpercaya|amanah',
    'akun': r'login|log in|daftar|otp|kode verifikasi|\bpin\b|verifikasi|wajah|ganti nomor|nomor hp|blokir|suspend'
            r'|dibekukan|premium|premier|upgrade|ktp|modifikasi',
    'error_performa': r'error|bug|gangguan|lambat|loading|\blag\b|tidak bisa dibuka|force close|keluar sendiri'
                      r'|maintenance|perbaikan|update|jaringan|koneksi|blank|crash',
    'cs': r'\bcs\b|call center|pusat bantuan|live agent|\bbot\b|pengaduan|komplain|customer|\bchat\b|email|tiket'
          r'|tidak (?:ada )?(?:respon|balas)',
    'biaya_promo': r'biaya|admin|potongan|potong|mahal|murah|gratis|cashback|voucher|promo|poin|koin|hadiah|event'
                   r'|iklan|tarif',
    'paylater': r'cicil|paylater|pinjam|limit|tagihan|jatuh tempo|denda|tenor|bpkb',
    'kemudahan': r'mudah|gampang|simple|simpel|praktis|ribet|rumit|bingung|tampilan|user friendly'
                 r'|fitur(?:nya)? lengkap',
}
Y_dev = dev[ASPEK].to_numpy()


def macro_f1_aspek(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)


hasil_aspek = []


def ringkas_aspek(nama, y_pred):
    """Presisi/recall/F1 per aspek + macro-F1 dengan interval bootstrap."""
    p, r, f, _ = precision_recall_fscore_support(Y_dev, y_pred, average=None, zero_division=0)
    bawah, atas = interval_bootstrap(Y_dev, y_pred, macro_f1_aspek)
    hasil_aspek[:] = [h for h in hasil_aspek if h['model'] != nama]
    hasil_aspek.append({
        'model': nama, 'macro_f1': round(macro_f1_aspek(Y_dev, y_pred), 3), 'ci_95': f'{bawah:.2f}-{atas:.2f}',
        **{f'f1_{a}': round(v, 3) for a, v in zip(ASPEK, f)},
    })
    print(f'{nama}: macro-F1 {macro_f1_aspek(Y_dev, y_pred):.3f} (CI 95% {bawah:.2f}-{atas:.2f}), '
          f'micro-F1 {f1_score(Y_dev, y_pred, average="micro", zero_division=0):.3f}')
    return pd.DataFrame({'presisi': p, 'recall': r, 'f1': f}, index=ASPEK).round(2)


pred_kata_kunci = np.column_stack([dev['teks_ringan'].str.contains(POLA_ASPEK[a], regex=True) for a in ASPEK]).astype(int)
tabel_kata_kunci = ringkas_aspek('Kata kunci', pred_kata_kunci)

Kata kunci: macro-F1 0.737 (CI 95% 0.70-0.76), micro-F1 0.740


In [12]:
def model_aspek(C):
    return Pipeline([
        ('vektor', buat_vektorisasi(min_df=2)),
        ('model', OneVsRestClassifier(LogisticRegression(C=C, class_weight='balanced', max_iter=2000))),
    ])


lipatan_aspek = KFold(n_splits=5, shuffle=True, random_state=SEED)
pred_aspek, skor_aspek = {}, {}
for C in [1, 4, 16]:
    pred = np.zeros_like(Y_dev)
    for latih, uji in lipatan_aspek.split(dev):
        data_latih = pd.concat([dev.iloc[latih], tambahan])
        model = model_aspek(C).fit(data_latih['teks_penuh'], data_latih[ASPEK].to_numpy())
        pred[uji] = model.predict(dev['teks_penuh'].iloc[uji])
    pred_aspek[C], skor_aspek[C] = pred, macro_f1_aspek(Y_dev, pred)
    print(f'C={C}: macro-F1 5-fold {skor_aspek[C]:.3f}')

C_ASPEK = max(skor_aspek, key=skor_aspek.get)
tabel_tfidf = ringkas_aspek(f'TF-IDF + OvR LR (C={C_ASPEK})', pred_aspek[C_ASPEK])

display(pd.concat({'kata_kunci': tabel_kata_kunci, 'tfidf_ovr_lr': tabel_tfidf}, axis=1)
        .assign(jumlah_dev=Y_dev.sum(axis=0)))

# Model final dilatih dengan seluruh gold train, untuk dinilai di test nanti
joblib.dump(model_aspek(C_ASPEK).fit(gold_train['teks_penuh'], gold_train[ASPEK].to_numpy()),
            FOLDER_MODEL / 'tfidf_aspek.joblib')
pd.DataFrame(hasil_aspek).assign(versi_label=VERSI_LABEL).to_parquet(
    FOLDER_PROCESSED / 'ringkasan_dev_baseline_aspek.parquet', index=False  # dibaca 04
)
print('Model tersimpan di models/: tfidf_sentimen_rating, tfidf_sentimen_gold, tfidf_aspek')

C=1: macro-F1 5-fold 0.756
C=4: macro-F1 5-fold 0.769
C=16: macro-F1 5-fold 0.770
TF-IDF + OvR LR (C=16): macro-F1 0.770 (CI 95% 0.74-0.80), micro-F1 0.790


kata_kunci              tfidf_ovr_lr              jumlah_dev
                  presisi recall    f1      presisi recall    f1           
transaksi            0.66   0.67  0.66         0.77   0.79  0.78        203
keamanan             0.79   0.52  0.63         0.73   0.60  0.66         50
akun                 0.88   0.73  0.80         0.87   0.77  0.81        137
error_performa       0.89   0.69  0.78         0.84   0.78  0.81        196
cs                   0.80   0.69  0.74         0.83   0.75  0.79         52
biaya_promo          0.69   0.86  0.77         0.93   0.77  0.84         87
paylater             0.89   0.89  0.89         0.97   0.80  0.88         70
kemudahan            0.54   0.78  0.64         0.70   0.51  0.59         45

Model tersimpan di models/: tfidf_sentimen_rating, tfidf_sentimen_gold, tfidf_aspek
